# 01 — Lookup de Municípios

**Objetivo:** criar uma tabela de bridge (lookup) que conecta o **código IBGE de 7 dígitos** presente nos microdados do ENEM com os **nomes e siglas de UF** usados no Atlas Brasil.

Essa tabela é necessária para futuros joins em nível municipal — quando dados municipais do IDHM/PNAD estiverem disponíveis, o notebook 04 os priorizará automaticamente via `COALESCE`.

**Input:** `datasets/códigos municipais/RELATORIO_DTB_BRASIL_2024_MUNICIPIOS.ods`  
**Output:** `data/processed/lookup/lookup_municipios.parquet`  
**Colunas geradas:** `uf_cod`, `nome_municipio`, `code_muni`, `sg_uf`, `nome_norm`, `uf_norm`

## 1. Imports e configuração

In [ ]:
import pandas as pd
import unicodedata, re
from pathlib import Path

Path('../data/processed/lookup').mkdir(parents=True, exist_ok=True)


## 2. Função de normalização de texto

Usada para padronizar nomes antes de comparações — remove acentos, coloca em minúsculo e elimina caracteres especiais. Será aplicada tanto no lookup quanto no Atlas Brasil para garantir que strings como `"São Paulo"` e `"sao paulo"` sejam tratadas como iguais em futuros joins fuzzy.

In [ ]:
def normaliza(nome):
    nome = unicodedata.normalize('NFKD', str(nome))
    nome = nome.encode('ASCII', 'ignore').decode()
    nome = nome.lower().strip()
    nome = re.sub(r'[^a-z0-9 ]', '', nome)
    return nome


## 3. Leitura do arquivo ODS (IBGE 2024)

O arquivo possui 6 linhas de cabeçalho burocrático antes dos dados reais — por isso `skiprows=6`.  
Colunas relevantes: `UF` (código numérico), `Nome_Município`, `Código Município Completo` (7 dígitos IBGE).

In [ ]:
df = pd.read_excel(
    '../datasets/códigos municipais/RELATORIO_DTB_BRASIL_2024_MUNICIPIOS.ods',
    engine='odf', skiprows=6
)
print(df.columns.tolist())
df.head(3)


## 4. Construção do lookup

O arquivo ODS usa o **código numérico** do estado (ex: 35 = SP), mas o Atlas Brasil usa **siglas** (SP, RJ...).  
Criamos o dicionário `UF_SIGLA` para fazer essa conversão, cobrindo todos os 27 estados + DF.

A coluna `nome_norm` aplica a função `normaliza()` ao nome do município — necessária para fuzzy matching futuro com os nomes do Atlas Brasil que aparecem no formato `"Abadia de Goiás (GO)"`.

> **Nota:** o código IBGE de 7 dígitos (`code_muni`) é a chave de join com a coluna `CO_MUNICIPIO_ESC` dos microdados do ENEM.

In [ ]:
UF_SIGLA = {
    11:'RO', 12:'AC', 13:'AM', 14:'RR', 15:'PA', 16:'AP', 17:'TO',
    21:'MA', 22:'PI', 23:'CE', 24:'RN', 25:'PB', 26:'PE',
    27:'AL', 28:'SE', 29:'BA',
    31:'MG', 32:'ES', 33:'RJ', 35:'SP',
    41:'PR', 42:'SC', 43:'RS',
    50:'MS', 51:'MT', 52:'GO', 53:'DF'
}

lookup = df[['UF', 'Nome_Município', 'Código Município Completo']].copy()
lookup = lookup.dropna(subset=['Código Município Completo'])
lookup.columns = ['uf_cod', 'nome_municipio', 'code_muni']
lookup['code_muni'] = lookup['code_muni'].astype(int)
lookup['uf_cod']    = lookup['uf_cod'].astype(int)
lookup['sg_uf']     = lookup['uf_cod'].map(UF_SIGLA)
lookup['nome_norm'] = lookup['nome_municipio'].apply(normaliza)
lookup['uf_norm']   = lookup['sg_uf'].str.upper()

print(f'Total: {len(lookup)} municípios')
lookup.head()


## 5. Exportação

In [ ]:
lookup.to_parquet('../data/processed/lookup/lookup_municipios.parquet', index=False)
print('Salvo: data/processed/lookup/lookup_municipios.parquet')
